# 07. 이터레이터 / 제너레이터

`for`문이 내부적으로 어떻게 동작하는지(이터레이터 프로토콜), 그리고 메모리를 아끼면서 대용량 데이터를 다루는 제너레이터. PyTorch `DataLoader`가 내부적으로 쓰는 패턴과 비슷합니다.

## 이터러블(iterable) vs 이터레이터(iterator)

`for x in obj`가 동작하려면 `obj`가 `__iter__`를 구현해야 하고, 그게 반환하는 이터레이터는 `__next__`를 구현해야 합니다. 소진되면 `StopIteration`을 던집니다.

In [1]:
nums = [1, 2, 3]
it = iter(nums)       # 리스트(iterable) -> 이터레이터
print(next(it))
print(next(it))
print(next(it))
try:
    next(it)
except StopIteration:
    print("더 이상 꺼낼 게 없음")


1
2
3
더 이상 꺼낼 게 없음


## 직접 이터레이터 클래스 만들기

In [2]:
class CountUp:
    def __init__(self, limit):
        self.limit = limit
        self.current = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.current >= self.limit:
            raise StopIteration
        self.current += 1
        return self.current

for n in CountUp(5):
    print(n, end=" ")
print()


1 2 3 4 5 


## 제너레이터 함수: yield

`yield`가 있는 함수는 호출해도 즉시 실행되지 않고 **제너레이터 객체**를 반환합니다. `next()`를 호출할 때마다 `yield` 지점까지만 실행되고 멈춥니다 — 위의 클래스와 같은 동작을 훨씬 짧게 구현합니다.

In [3]:
def count_up(limit):
    current = 0
    while current < limit:
        current += 1
        yield current

for n in count_up(5):
    print(n, end=" ")
print()

gen = count_up(3)
print(next(gen), next(gen), next(gen))


1 2 3 4 5 
1 2 3


## 메모리 효율: 리스트 대신 제너레이터

리스트 컴프리헨션은 전체를 메모리에 만들지만, 제너레이터 표현식(`()`로 감싸기)은 필요할 때마다 값을 '하나씩' 만듭니다. 대용량 로그/데이터셋을 스트리밍 처리할 때 핵심 패턴입니다.

In [4]:
import sys

list_comp = [x * x for x in range(100000)]       # 전부 메모리에 올라감
gen_expr = (x * x for x in range(100000))        # 값 하나 만들 준비만 됨

print("list 크기:", sys.getsizeof(list_comp), "bytes")
print("generator 크기:", sys.getsizeof(gen_expr), "bytes")

print(sum(gen_expr))  # 합계를 구할 때만 순차적으로 값이 생성됨 (메모리 절약)


list 크기: 800984 bytes
generator 크기: 200 bytes
333328333350000


## itertools: 제너레이터를 다루는 표준 도구상자

In [5]:
import itertools

# islice: 제너레이터에서 슬라이싱 (무한 제너레이터에도 쓸 수 있음)
def infinite_counter():
    n = 0
    while True:
        yield n
        n += 1

first_five = list(itertools.islice(infinite_counter(), 5))
print(first_five)

# chain: 여러 이터러블을 하나처럼 이어붙이기
combined = list(itertools.chain([1, 2], [3, 4], [5]))
print(combined)


[0, 1, 2, 3, 4]
[1, 2, 3, 4, 5]


## 헷갈렸던 점 / 더 찾아볼 것

- 제너레이터는 **한 번 소진하면 재사용 불가** — 다시 쓰려면 함수를 다시 호출해 새 제너레이터를 만들어야 함
- PyTorch `Dataset.__getitem__` + `DataLoader`도 결국 이터레이터 프로토콜을 감싼 것 — 이 개념이 익숙해지면 커스텀 데이터 파이프라인 작성이 쉬워짐